In [ ]:
from IPython.display import Latex
from labelstudio.LabelStudioInterface import LabelStudioInterface
from IPython.display import clear_output
from preprocessing.helpers.helper_to_classes import get_image_path_from_task
from preprocessing.AnnotatedPage import AnnotatedPage
from PIL import Image

LS = LabelStudioInterface()
LS_export_data = LS.raw_tasks

#cajas revisadas hasta la 341.
max_revised = 125 #poner aquí la página por la que se está revisando.

INVERSE_ORDER = False

tasks = [task for task in sorted(LS_export_data, key = lambda task : task["id"]) if task["id"] > max_revised]

if INVERSE_ORDER:
    tasks = tasks[::-1]



for task in tasks:
    task_id = task["id"]

    img = Image.open(get_image_path_from_task(task))


    for (k,ann) in enumerate(task["annotations"]):
        Ann = AnnotatedPage(ann, img, unrotate = True, cc_ordering = True)
        print(f"-- -- Llevas un {len([x for x in LS.raw_tasks if x["id"] <= task_id]) / len(LS_export_data) * 100:.2f}%")
        print(f"-- -- TAREA {task_id} -- ANOTACIÓN {k} --")
        print(f"-- -- --       Completada por: {Ann.completer:<25}")
        print(f"-- -- -- Último actualización: {Ann.updater:<25} con fecha de {Ann.last_update_time}")
        boxes_by_height = sorted(list(Ann.image_boxes.values()), key = lambda box: (box.polygon.centroid.y, box.polygon.centroid.x))


        for (i, box) in enumerate(boxes_by_height):
            print(f"\t ({task_id}) Caja {repr(box)}:")
            #no generamos el recorte tal cual, sino que lo metemos
            #en un rectángulo (esto hace más fácil leer)
            display(Ann.generate_collage([box.id]))
            display(Latex(box.associated_fragments[0].text))
            print("\n")

        a = input("Continuamos? (vacío para continuar)")
        if a.lower() != "":
            raise KeyboardInterrupt("Se ha parado el proceso")

        clear_output(wait = True)